# The Proper EDA: Lending Club Loan Data (2007-2018)

A deep exploratory analysis of 2.26 million consumer loans originated through Lending Club. This notebook treats EDA as the product - not a quick pass before modeling, but a rigorous, structured investigation of data quality, distributional properties, temporal dynamics, multivariate structure, and portfolio-level credit analytics.

**Dataset:** [Lending Club Loan Data](https://www.kaggle.com/datasets/wordsforthewise/lending-club) - 2.26M loans, 151 features, 2007-2018.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, leaves_list
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
# -- Color Palette --
C = {
    "primary":    "#1e293b",
    "secondary":  "#334155",
    "accent":     "#2563eb",
    "accent2":    "#7c3aed",
    "risk":       "#dc2626",
    "safe":       "#059669",
    "warn":       "#d97706",
    "light_gray": "#f1f5f9",
    "mid_gray":   "#94a3b8",
    "dark_gray":  "#475569",
    "bg":         "#ffffff",
}

# Grade color scale (A=safest green through G=riskiest red)
GRADE_COLORS = {
    "A": "#059669", "B": "#10b981", "C": "#d97706",
    "D": "#f59e0b", "E": "#ef4444", "F": "#dc2626", "G": "#991b1b",
}

# Sequential blue palette for heatmaps
BLUE_CMAP = LinearSegmentedColormap.from_list("blue_seq", ["#f0f9ff", "#2563eb", "#1e3a5f"])
# Diverging palette
DIV_CMAP = LinearSegmentedColormap.from_list("div", ["#059669", "#f1f5f9", "#dc2626"])

# -- Matplotlib Style --
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#ffffff",
    "axes.edgecolor": "#e2e8f0",
    "axes.labelcolor": C["primary"],
    "axes.titlecolor": C["primary"],
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.color": "#f1f5f9",
    "grid.linewidth": 0.8,
    "xtick.color": C["dark_gray"],
    "ytick.color": C["dark_gray"],
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial"],
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "legend.frameon": False,
    "legend.fontsize": 9,
})

def style_axis(ax, title="", subtitle="", xlabel="", ylabel=""):
    """Apply consistent styling to a matplotlib axis."""
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold", color=C["primary"], pad=12, loc="left")
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10,
                color=C["dark_gray"], style="italic", va="bottom")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11, color=C["secondary"])
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11, color=C["secondary"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#e2e8f0")
    ax.spines["bottom"].set_color("#e2e8f0")
    return ax

def fmt_pct(x, _=None):
    return f"{x:.0%}" if abs(x) < 10 else f"{x:.1%}"

def fmt_k(x, _=None):
    if abs(x) >= 1e6:
        return f"{x/1e6:.1f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"

print("Style configured.")

In [ ]:
df = pd.read_csv("data/accepted_2007_to_2018Q4.csv.gz", low_memory=False)

# Parse dates
df["issue_d"] = pd.to_datetime(df["issue_d"], format="mixed")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="mixed", errors="coerce")

# Derive key columns upfront
df["issue_year"] = df["issue_d"].dt.year
df["issue_month"] = df["issue_d"].dt.to_period("M")
df["credit_history_years"] = (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25

# Binary default indicator (Charged Off or Default = 1, else 0)
default_statuses = ["Charged Off", "Default"]
df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)

print(f"Loaded: {df.shape[0]:,} loans x {df.shape[1]} features")
print(f"Date range: {df['issue_d'].min():%Y-%m} to {df['issue_d'].max():%Y-%m}")
print(f"Default rate: {df['is_default'].mean():.2%}")

---
## 1. Data Audit

Before any analysis, understand what you have: types, cardinality, memory footprint, duplicates, and structural organization of the 151 features.

In [ ]:
mem_usage = df.memory_usage(deep=True)
total_mb = mem_usage.sum() / 1e6

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total memory: {total_mb:,.1f} MB")
print(f"\nDtype counts:")
print(df.dtypes.value_counts().to_string())

In [ ]:
col_summary = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.count(),
    "null_pct": df.isnull().mean(),
    "n_unique": df.nunique(),
    "pct_unique": df.nunique() / len(df),
    "top_value": df.apply(lambda x: x.value_counts().index[0] if x.count() > 0 else None),
    "top_freq_pct": df.apply(lambda x: x.value_counts(normalize=True).iloc[0] if x.count() > 0 else None),
    "memory_mb": df.memory_usage(deep=True)[1:] / 1e6,
})
col_summary = col_summary.sort_values("memory_mb", ascending=False)

print("Top 30 columns by memory usage:")
print(col_summary.head(30).to_string())

In [ ]:
obj_cols = df.select_dtypes(include="object").columns
dtype_issues = []
for col in obj_cols:
    sample = df[col].dropna().head(1000)
    numeric_frac = pd.to_numeric(sample.str.replace("[%,$]", "", regex=True), errors="coerce").notna().mean()
    if numeric_frac > 0.8:
        dtype_issues.append({"column": col, "numeric_fraction": numeric_frac, "sample_values": list(sample.head(3))})

if dtype_issues:
    print("Columns stored as object but appear numeric:")
    for d in dtype_issues:
        print(f"  {d['column']}: {d['numeric_fraction']:.0%} numeric | samples: {d['sample_values']}")
else:
    print("No dtype mismatches found.")

In [ ]:
card = col_summary[["n_unique", "pct_unique", "top_freq_pct"]].copy()
card = card.sort_values("n_unique", ascending=True).tail(40)

colors_bar = []
for idx in card.index:
    if card.loc[idx, "n_unique"] <= 2 or card.loc[idx, "top_freq_pct"] > 0.99:
        colors_bar.append(C["risk"])
    elif card.loc[idx, "n_unique"] > 1000:
        colors_bar.append(C["warn"])
    else:
        colors_bar.append(C["accent"])

fig, ax = plt.subplots(figsize=(12, 14))
ax.barh(range(len(card)), card["n_unique"], color=colors_bar, height=0.7)
ax.set_yticks(range(len(card)))
ax.set_yticklabels(card.index, fontsize=8)
ax.set_xscale("log")
style_axis(ax, title="Feature Cardinality (Top 40)",
           subtitle="Red = near-constant, Amber = high cardinality (>1000), Blue = normal",
           xlabel="Unique Values (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
near_const = col_summary[(col_summary["n_unique"] <= 2) | (col_summary["top_freq_pct"] > 0.99)]
if len(near_const) > 0:
    print(f"Near-constant columns ({len(near_const)}):")
    print(near_const[["n_unique", "top_value", "top_freq_pct"]].to_string())
else:
    print("No near-constant columns found.")

In [ ]:
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes:,} ({exact_dupes/len(df):.4%})")

near_dupe_cols = ["loan_amnt", "funded_amnt", "int_rate", "grade", "issue_d", "addr_state"]
available = [c for c in near_dupe_cols if c in df.columns]
near_dupes = df.duplicated(subset=available).sum()
print(f"Near-duplicate rows (same {', '.join(available)}): {near_dupes:,} ({near_dupes/len(df):.2%})")

In [ ]:
COLUMN_GROUPS = {
    "Loan Terms": ["id", "member_id", "loan_amnt", "funded_amnt", "funded_amnt_inv", "term", "int_rate",
                    "installment", "grade", "sub_grade", "url", "desc"],
    "Borrower Demographics": ["emp_title", "emp_length", "home_ownership", "annual_inc",
                               "verification_status", "addr_state", "zip_code",
                               "annual_inc_joint", "dti_joint", "verification_status_joint"],
    "Credit History": ["dti", "delinq_2yrs", "earliest_cr_line", "fico_range_low",
                       "fico_range_high", "inq_last_6mths", "open_acc", "pub_rec",
                       "revol_bal", "revol_util", "total_acc", "pub_rec_bankruptcies",
                       "mort_acc", "num_actv_bc_tl", "num_bc_sats", "num_bc_tl",
                       "num_il_tl", "num_op_rev_tl", "num_rev_accts", "num_rev_tl_bal_gt_0",
                       "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd",
                       "num_tl_90g_dpd_24m", "num_tl_op_past_12m",
                       "pct_tl_nvr_dlq", "percent_bc_gt_75",
                       "tot_hi_cred_lim", "total_bal_ex_mort", "total_bc_limit",
                       "total_il_high_credit_limit", "acc_open_past_24mths",
                       "avg_cur_bal", "bc_open_to_buy", "bc_util",
                       "chargeoff_within_12_mths", "collections_12_mths_ex_med",
                       "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op",
                       "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl",
                       "mths_since_last_delinq", "mths_since_last_major_derog",
                       "mths_since_last_record", "mths_since_recent_bc",
                       "mths_since_recent_bc_dlq", "mths_since_recent_inq",
                       "mths_since_recent_revol_delinq", "tax_liens",
                       "tot_coll_amt", "tot_cur_bal", "total_rev_hi_lim",
                       "last_fico_range_high", "last_fico_range_low",
                       "acc_now_delinq", "delinq_amnt",
                       "num_accts_ever_120_pd", "num_actv_rev_tl",
                       "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m",
                       "mths_since_rcnt_il", "total_bal_il", "il_util",
                       "open_rv_12m", "open_rv_24m", "max_bal_bc", "all_util",
                       "inq_fi", "total_cu_tl", "inq_last_12m"],
    "Loan Status & Payments": ["loan_status", "pymnt_plan", "issue_d", "purpose", "title",
                                "initial_list_status", "application_type", "disbursement_method",
                                "out_prncp", "out_prncp_inv", "total_pymnt",
                                "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
                                "total_rec_late_fee", "recoveries",
                                "collection_recovery_fee", "last_pymnt_d",
                                "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
                                "policy_code"],
    "Hardship & Settlement": ["hardship_flag", "hardship_type", "hardship_reason",
                               "hardship_status", "hardship_amount",
                               "hardship_start_date", "hardship_end_date",
                               "hardship_length", "hardship_dpd",
                               "hardship_loan_status", "hardship_payoff_balance_amount",
                               "hardship_last_payment_amount",
                               "deferral_term", "payment_plan_start_date",
                               "orig_projected_additional_accrued_interest",
                               "debt_settlement_flag", "debt_settlement_flag_date",
                               "settlement_status", "settlement_date",
                               "settlement_amount", "settlement_percentage",
                               "settlement_term"],
    "Secondary Applicant": ["revol_bal_joint",
                             "sec_app_fico_range_low", "sec_app_fico_range_high",
                             "sec_app_earliest_cr_line", "sec_app_inq_last_6mths",
                             "sec_app_mort_acc", "sec_app_open_acc",
                             "sec_app_revol_util", "sec_app_open_act_il",
                             "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths",
                             "sec_app_collections_12_mths_ex_med",
                             "sec_app_mths_since_last_major_derog"],
}

for group, cols in COLUMN_GROUPS.items():
    present = [c for c in cols if c in df.columns]
    missing_from_df = [c for c in cols if c not in df.columns]
    print(f"\n{group}: {len(present)} features present" + (f", {len(missing_from_df)} not in data" if missing_from_df else ""))

uncategorized = set(df.columns) - set(c for cols in COLUMN_GROUPS.values() for c in cols)
derived = {"issue_year", "issue_month", "credit_history_years", "is_default"}
uncategorized -= derived
if uncategorized:
    print(f"\nUncategorized ({len(uncategorized)}): {sorted(uncategorized)}")

In [ ]:
print("Memory optimization recommendations:\n")
opt_savings = 0
for col in df.columns:
    if df[col].dtype == "float64":
        opt_savings += df[col].memory_usage(deep=True) * 0.5
    if df[col].dtype == "object" and df[col].nunique() < 50:
        current = df[col].memory_usage(deep=True)
        as_cat = df[col].astype("category").memory_usage(deep=True)
        opt_savings += current - as_cat

print(f"Estimated memory savings from downcasting: ~{opt_savings/1e6:,.0f} MB")
print(f"Current total: {df.memory_usage(deep=True).sum()/1e6:,.0f} MB")
print(f"Potential total: {(df.memory_usage(deep=True).sum() - opt_savings)/1e6:,.0f} MB")